<p style="color:#FFF; background:#07D; padding:12px; font-size:20px; font-style:italic; text-align:center">
<span style="width:49%; display:inline-block; text-align:left">Christophe Schlick</span>
<span style="width:49%; display:inline-block; text-align:right">schlick[at]u-bordeaux.fr</span>
<span style="font-size:48px; font-style:normal"><b>EXTRACTION&ensp;DE&ensp;DONNÉES</b></span><br>
<span style="width:49%; display:inline-block; text-align:left">Version 2024-12</span>
<span style="width:49%; display:inline-block; text-align:right">Licence CC-BY-NC-ND</span></p>

De nos jours, la plupart des jeux de données utilisés dans le domaine des Sciences des Données sont disponibles sur le web. Avec l'avènement du mouvement ***Open Data*** depuis une dizaine d'années, ces jeux de données sont regroupés sur des sites dédiés (j'en ai listé un certain nombre dans la page consacrée au sujet de projet) et facilement téléchargeables dans des formats standardisés, les plus fréquent étant les formats **CSV**, **JSON** et **XML**.

Néanmoins, il existe de très nombreuses données qui ne sont pas explicitement formatés pour être facilement téléchargées, la plupart du temps parce que ***leurs auteurs ont organisé ces données pour faciliter leur lecture sur une page web, mais pas forcément leur traitement par des outils d'automatisation***. Lorsqu'on souhaite récupérer et structurer ce type de données, il faut mettre en oeuvre des outils spécifiques, qui se regroupent dans deux catégories principales :

- Les outils de **web scraping** (en français, ***gratter ou racler le web***) permettent d'identifier et de récupérer des données spécifiques au sein d'une page web
- Les outils de **web crawling** (en français, ***ramper ou crapahuter sur le web***) permettent de naviguer récursivement (soit en largeur, soit en profondeur) à partir d'une page web, en suivant certains des liens qui sont définis sur cette page

Les deux techniques combinées permettent de récupérer une énorme masse de données : globalement tout ce qui est affichable sur le web est récupérable avec ce type d'outils d'automatisation. L'objectif de ce chapitre est de lister quelques techniques simples pour réaliser cette extraction de données.

In [1]:
from SRC.tools import show, load, fetch
import pandas as pd

<h2 style="padding:16px; color:#FFF; background:#07D">A - load, fetch et pandas</h2>

### 1 - Données au format TXT

Dans le chapitre 3, nous avons détaillé l'utilisation de la fonction **`load`** du module **`tools`** qui permet de récupérer le contenu de n'importe quel fichier texte, aussi bien dans un dossier stocké en local sur le disque de l'utilisateur, que sur un serveur distant, via une URL indiquant l'adresse du fichier à télécharger. Par défaut la fonction **`load`** va découper le contenu du fichier ligne par ligne, supprimer les lignes vides et les lignes de commentaires, et enfin retourner les lignes restantes sous la forme d'une liste de chaînes de caractères, ce qui est le traitement le plus intéressant pour les fichiers semi-structurés :

In [2]:
url = 'https://www.labri.fr/perso/schlick/outinfo/TEST/test-TXT.txt'

lines = load(url) # lecture des données avec suppression des commentaires et un 'split' à chaque ligne
print(lines) # les données sont structurées sous forme d'une liste de lignes

['Nom         Hugo', 'Prénom      Victor', 'Naissance   26/02/1802 Besançon', 'Décès       22/05/1885 Paris', 'Nom         Baudelaire', 'Prénom      Charles', 'Naissance   09/04/1821 Paris', 'Décès       31/08/1867 Paris', 'Nom         Rimbaud', 'Prénom      Arthur', 'Naissance   20/10/1854 Charleville', 'Décès       10/11/1891 Marseille']


Mais si nécessaire, on peut garder le contenu au format brut en modifiant le paramètre **`split`** de la fonction :

In [3]:
text = load(url, split='') # lecture des données au format brut (= préservation du contenu intégral)
print(text) # les données sont structurées sous forme d'une chaîne multi-lignes

#
# Auteurs français du XIXe siècle
#

Nom         Hugo
Prénom      Victor
Naissance   26/02/1802 Besançon
Décès       22/05/1885 Paris

Nom         Baudelaire
Prénom      Charles
Naissance   09/04/1821 Paris
Décès       31/08/1867 Paris

Nom         Rimbaud
Prénom      Arthur
Naissance   20/10/1854 Charleville
Décès       10/11/1891 Marseille


---
### 2 - Données aux formats CSV et JSON

La même fonction **`load`** permet de lire des **fichiers CSV** en appliquant un léger post-traitement :

In [4]:
url = 'https://www.labri.fr/perso/schlick/outinfo/TEST/test-CSV.csv'

lines = load(url) # lecture des données CSV avec suppression des commentaires
print(lines, '\n') # les données sont structurées sous forme d'une liste de lignes CSV
csv = [line.split(', ') for line in lines] # découpage de chaque ligne CSV selon les virgules
print(csv) # les données sont maintenant structurées sous forme d'une liste de listes

['Nom, Prénom, NaissanceDate, NaissanceLieu, DécèsDate, DécèsLieu', 'Hugo, Victor, 26/02/1802, Besançon, 22/05/1885, Paris', 'Baudelaire, Charles, 09/04/1821, Paris, 31/08/1867, Paris', 'Rimbaud, Arthur, 20/10/1854, Charleville, 10/11/1891, Marseille'] 

[['Nom', 'Prénom', 'NaissanceDate', 'NaissanceLieu', 'DécèsDate', 'DécèsLieu'], ['Hugo', 'Victor', '26/02/1802', 'Besançon', '22/05/1885', 'Paris'], ['Baudelaire', 'Charles', '09/04/1821', 'Paris', '31/08/1867', 'Paris'], ['Rimbaud', 'Arthur', '20/10/1854', 'Charleville', '10/11/1891', 'Marseille']]


Mais comme on l'a vu au chapitre 7, la fonction **`read_csv`** de **pandas** est bien plus simple pour ce genre de traitement :

In [5]:
table = pd.read_csv(url, comment='#') # il faut définir le caractère préfixe utilisé pour les commentaires
table # les données sont structurées sous forme d'une table 'pandas'

,Nom,Prénom,NaissanceDate,NaissanceLieu,DécèsDate,DécèsLieu
0,Hugo,Victor,26/02/1802,Besançon,22/05/1885,Paris
1,Baudelaire,Charles,09/04/1821,Paris,31/08/1867,Paris
2,Rimbaud,Arthur,20/10/1854,Charleville,10/11/1891,Marseille


Le processus de lecture et de conversion est très similaire pour les **fichiers JSON** :

In [6]:
url = 'https://www.labri.fr/perso/schlick/outinfo/TEST/test-JSON.json'

json = load(url, split='') # lecture des données JSON au format brut
print(json, '\n') # les données sont formatées sous la forme d'une chaîne JSON multi-lignes
json = eval(json, {}, {}) # transformation de la chaîne en liste de dictionnaires avec la fonction 'eval'
print(json) # les données sont maintenant structurées sous forme d'une liste de dictionnaires

[
  {
    "Nom" : "Hugo",
    "Prénom" : "Victor",
    "Naissance" : "26/02/1802 Besançon",
    "Décès" : "22/05/1885 Paris"
  },
  {
    "Nom" : "Baudelaire",
    "Prénom" : "Charles",
    "Naissance" : "09/04/1821 Paris",
    "Décès" : "31/08/1867 Paris"
  },
  {
    "Nom" : "Rimbaud",
    "Prénom" : "Arthur",
    "Naissance" : "20/10/1854 Charleville",
    "Décès" : "10/11/1891 Marseille"
  }
] 

[{'Nom': 'Hugo', 'Prénom': 'Victor', 'Naissance': '26/02/1802 Besançon', 'Décès': '22/05/1885 Paris'}, {'Nom': 'Baudelaire', 'Prénom': 'Charles', 'Naissance': '09/04/1821 Paris', 'Décès': '31/08/1867 Paris'}, {'Nom': 'Rimbaud', 'Prénom': 'Arthur', 'Naissance': '20/10/1854 Charleville', 'Décès': '10/11/1891 Marseille'}]


Mais là encore, on a vu que la fonction **`read_json`** de **pandas** est bien plus simple pour ce genre de traitement :

In [7]:
table = pd.read_json(url) # pas de préfixe pour identifier les commentaires, car ils n'existent pas en JSON
table # les données sont structurées sous forme d'une table 'pandas'

,Nom,Prénom,Naissance,Décès
0,Hugo,Victor,26/02/1802 Besançon,22/05/1885 Paris
1,Baudelaire,Charles,09/04/1821 Paris,31/08/1867 Paris
2,Rimbaud,Arthur,20/10/1854 Charleville,10/11/1891 Marseille


---
### 3 - Données aux formats XML ou HTML

Pour les fichiers XML, la fonction **`load`** ne permet que de récupérer le code source. Pour effectuer l'analyse du contenu, notamment lorsque la structure du fichier contient de nombreux niveaux hiérarchiques, l'outil idéal est la bibliothèque **BeautifulSoup** qui sera étudiée dans la section C, plus bas :

In [8]:
url = 'https://www.labri.fr/perso/schlick/outinfo/TEST/test-XML.xml'

xml = load(url, split='') # lecture des données XML au format brut
print(xml) # les données sont structurées sous forme d'une chaîne XML multi-lignes

<?xml version="1.0" encoding="UTF-8" ?>

<!-- Auteurs français du XIXe siècle -->

<auteurs>
  <auteur Nom="Hugo" Prénom="Victor"
    Naissance="26/02/1802 Besançon" Décès="22/05/1885 Paris" />
  <auteur Nom="Baudelaire" Prénom="Charles"
    Naissance="09/04/1821 Paris" Décès="31/08/1867 Paris" />
  <auteur Nom="Rimbaud" Prénom="Arthur"
    Naissance="20/10/1854 Charleville" Décès="10/11/1891 Marseille" />
</auteurs>


Lorsque le fichier XML ne contient que des données matricielles et non hiérarchiques (ce qui est le cas ici), la fonction **`read_xml`** de **pandas** permet de faire cette conversion directement :

In [9]:
table = pd.read_xml(url) # lecture des données XML et suppression automatique des commentaires
table # les données sont structurées sous forme d'une table 'pandas'

,Nom,Prénom,Naissance,Décès
0,Hugo,Victor,26/02/1802 Besançon,22/05/1885 Paris
1,Baudelaire,Charles,09/04/1821 Paris,31/08/1867 Paris
2,Rimbaud,Arthur,20/10/1854 Charleville,10/11/1891 Marseille


Les mêmes remarques s'appliquent évidemment aux fichiers HTML, qui ne constituent qu'une variante du format XML :

In [10]:
url = 'https://www.labri.fr/perso/schlick/outinfo/TEST/test-HTML.html'

html = load(url, split='') # lecture des données HTML au format brut
print(html) # les données sont structurées sous forme d'une chaîne XML multi-lignes

<html><head><meta charset="utf-8"/>
<style>
body {font-family:arial;}
table {border:2px solid black; border-spacing:0px; text-align:center;}
th,td {border:1px solid black;}
</style></head>
<body>
<h2>Auteurs français du XIXe siècle</h2>
<table cellpadding=5>
<tr>
<th>Nom <th>Prénom
<th>NaissanceDate <th>NaissanceLieu
<th>DécèsDate <th>DécèsLieu
<tr>
<td>Hugo <td>Victor
<td>26/02/1802 <td>Besançon
<td>22/05/1885 <td>Paris
<tr>
<td>Baudelaire <td>Charles
<td>09/04/1821 <td>Paris
<td>31/08/1867 <td>Paris
<tr>
<td>Rimbaud <td>Arthur
<td>20/10/1854 <td>Charleville
<td>10/11/1891 <td>Marseille
</table>
</body></html>


Comme on l'a vu dans le chapitre 7, lorsqu'un fichier HTML contient des données structurées par les balises **`<table> ... </table>`** (ce qui est le cas ici), la fonction **`read_html`** de **pandas** permet de récupérer le contenu de toutes ces tables et de les retourner sous forme d'une liste de tables **pandas** :

In [11]:
tables = pd.read_html(url) # création d'une table 'pandas' pour chaque table HTML définie dans le fichier
tables[0] # affichage de la table d'indice 0 (c'est la seule qui est présente dans le fichier)

,Nom,Prénom,NaissanceDate,NaissanceLieu,DécèsDate,DécèsLieu
0,Hugo,Victor,26/02/1802,Besançon,22/05/1885,Paris
1,Baudelaire,Charles,09/04/1821,Paris,31/08/1867,Paris
2,Rimbaud,Arthur,20/10/1854,Charleville,10/11/1891,Marseille


Il est important de savoir qu'une partie non négligeable ***des sites web appliquent une politique anti-scraping***, en interdisant l'accès à leurs pages par des logiciels robots. Par exemple, le site **Transfermarkt** utilisé dans l'exercice G3, va bloquer l'utilisation de la fonction **`read_html`** de **pandas**, alors que la même page s'affiche sans difficulté si on charge directement cette page dans un navigateur web :

In [12]:
url = 'https://www.transfermarkt.fr/vereins-statistik/wertvollstemannschaften/marktwertetop'

#tables = pd.read_html(url) # HTTP error 403 : forbidden (= accès interdit hors navigateur)

Pour contourner ce blocage, le module **`tools`** contient une fonction **`fetch`** qui permet de faire croire au site que c'est bien un navigateur web et non un robot, qui est en train de se connecter. Il suffit ainsi de filter l'URL avec cette fonction **`fetch`** pour pouvoir utiliser la fonction **`read_html`** de manière classique :

In [13]:
tables = pd.read_html(fetch(url)) # utilisation de 'read_html' en filtrant l'URL avec 'fetch'
teams = tables[1] # extraction de la table d'indice 1 (c'est celle-là qui nous intéresse sur la page web)
teams.index = teams['#']; teams.index.name = None # transformation la colonne '#' en index
del teams['#']; del teams['Unnamed: 1'] # suppression des colonnes inutiles
teams[:10] # affichage des 10 premières lignes, pour vérification

,Club,Compétition,Valeur marchande
1,Real Madrid,LaLiga,"1,36 mrd. €"
2,Manchester City,Premier League,"1,26 mrd. €"
3,Arsenal FC,Premier League,"1,17 mrd. €"
4,Chelsea FC,Premier League,"963,20 mio. €"
5,FC Barcelone,LaLiga,"946,00 mio. €"
6,Bayern Munich,Bundesliga,"936,70 mio. €"
7,FC Liverpool,Premier League,"931,00 mio. €"
8,Paris Saint-Germain,Ligue 1,"892,50 mio. €"
9,Manchester United,Premier League,"854,15 mio. €"
10,Tottenham Hotspur,Premier League,"788,30 mio. €"


<h2 style="padding:16px; color:#FFF; background:#07D">B - requests</h2>

Lorsque l'interaction avec le serveur web nécessite plus de paramètres que la simple lecture d'une page web, l'association **`load + fetch + pandas`** ne suffit pas toujours. Dans ces cas, des outils plus évolués, tels que la bibliothèque **`requests`** permettent d'effecteur des requêtes HTTP plus complexes. Le package **`requests`** ne fait pas partie de la bibliothèque standard de Python, elle nécessite donc une installation spécifique à l'aide de l'outil **`pip`**, mais comme les autres packages très populaires dans le domaine des Sciences des Données, elle est directement incluse dans la distribution **Anaconda**. Et comme d'habitude également, la documentation complète du package se trouve sur le site [**ReadTheDocs**](https://requests.readthedocs.io/en/latest)

La plus-value principale de **requests** par rapport à la fonction **`load`** est la possibilité de gérer de manière fine les deux types de commandes impliquées dans le protocole HTTP : la commande **GET** pour récupérer des informations et la commande **POST** pour envoyer des informations. Voici quelques exemples d'utilisation :

In [14]:
import requests # import du package 'requests'

---
### 1 - Utilisation des requêtes GET

Pour simplifier l'utilisation des requêtes GET avec la bibliothèque **`requests`**, on va créer une fonction **`url_get`** qui va regrouper la séquence des opérations habituellement utilisées dans ce type de requête :

In [15]:
def url_get(url, **args):
  """apply GET request on provided URL, using optional arguments stored as arbitrary keyword arguments"""
  headers = {'User-Agent':'Mozilla/5.0'} # use Mozilla/Firefox user agent (anti-scraping filter)
  data = requests.get(url, headers=headers, params=args) # connect to provided URL and try to get data
  data.raise_for_status() # raise HTTPError exception when connection error or server error
  return data.json() # return data in JSON format when status is OK

On teste le principe des requêtes GET sur le site **`quotable.io/random`** qui est un générateur de citations aléatoires. A chaque requête GET qu'on envoie au site, il retourne une chaîne de caractères au format JSON qui contient une citation différentes, accompagnée de diverses informations complémentaire sur cette citation.

Dans une première version, on va utiliser la combinaison entre les fonctions **`load`** et **`eval`** comme on l'a fait dans la section A, pour récupérer et décoder la chaîne JSON retournée par le site :

In [16]:
url = 'http://api.quotable.io/random' # générateur de citations aléatoires

json = load(url, split='') # lecture des données au format brut renvoyées par le site
print(json, '\n') # les données sont formatées sous la forme d'une chaîne JSON multi-lignes
json = eval(json,{},{}) # conversion de la chaîne en dictionnaire avec la fonction 'eval'
print(f"● {json['content']} ({json['author']})") # extraction des champs 'content' et 'author'

{"_id":"3qfuatrZlFbw","content":"Experience keeps a dear school, but fools will learn in no other.","author":"Benjamin Franklin","tags":["Famous Quotes"],"authorSlug":"benjamin-franklin","length":65,"dateAdded":"2020-02-07","dateModified":"2023-04-14"} 

● Experience keeps a dear school, but fools will learn in no other. (Benjamin Franklin)


Dans une seconde version, grâce à l'utilisation de la fonction **`url_get`**, le processus va être grandement simplifié :

In [17]:
json = url_get(url) # lecture des données et stockage sous la forme d'un dictionaire JSON
print(f"● {json['content']} ({json['author']})") # extraction des champs 'content' et 'author'

● Don't turn away from possible futures before you're certain you don't have anything to learn from them. (Richard Bach)


---
L'intérêt principal des requêtes GET, est qu'elles permettent à l'utilisateur de transmettre des paramètres pour affiner sa demande. Cela peut s'effectuer de deux manières :

- soit on complète l'URL en rajoutant un motif de la forme **`?key=value&key=value&...`** pour fournir un certain nombre de couples ***(key,value)*** à la requête
- soit on transmet directement ces couples à la fonction qui effectue la requête sous la forme de paramètres nommés **`key=value, key=value,...`**

La première option est moins confortable car elle nécessite de gérer spécifiquement les caractères qui peuvent empêcher la bonne construction de l'URL : un certain nombre de caractères sont interdits par la norme, par exemple **`/ & % =`**, le plus gênant étant l'interdiction du caractère espace qui doit être remplacé par la séquence **`%20`** . Mais c'est la seule option disponible lorsqu'on utilise la fonction **`load`** :

In [18]:
# on construit l'URL pour inclure les deux contraintes : 5 citations dont l'auteur est Confucius
url = 'http://api.quotable.io/quotes?limit=5&author=Confucius'

json = load(url, split='') # lecture des données au format brut renvoyées par le site
print(json, '\n') # les données (complexes) sont formatées sous la forme d'une chaîne JSON multi-lignes
json = eval(json,{},{}) # conversion de la chaîne en dictionnaire avec la fonction 'eval'
for quote in json['results']: # les citations sont regroupées sous la clé 'results' du dictionnaire
  print(f"● {quote['content']} ({quote['author']})") # extraction des champs 'content' et 'author'

{"count":5,"totalCount":44,"page":1,"totalPages":9,"lastItemIndex":5,"results":[{"_id":"nH5op9VWSA5","author":"Confucius","content":"The superior man understands what is right; the inferior man understands what will sell.","tags":["Business"],"authorSlug":"confucius","length":88,"dateAdded":"2022-07-06","dateModified":"2023-04-14"},{"_id":"6Kl3UT6ULk","content":"Wisdom, compassion, and courage are the three universally recognized moral qualities of men.","author":"Confucius","tags":["Wisdom"],"authorSlug":"confucius","length":92,"dateAdded":"2021-05-12","dateModified":"2023-04-14"},{"_id":"Oh-e1-oygRPX","content":"To be wronged is nothing unless you continue to remember it.","author":"Confucius","tags":["Wisdom"],"authorSlug":"confucius","length":60,"dateAdded":"2021-05-12","dateModified":"2023-04-14"},{"_id":"LPtv5gxsvhsr","content":"If you look into your own heart, and you find nothing wrong there, what is there to worry about? What is there to fear?","author":"Confucius","tags":["Wi

La seconde option est beaucoup plus simple à mettre en oeuvre avec la fonction **`get_url`** :

In [19]:
url = 'http://api.quotable.io/quotes' # URL de base (pas la peine d'y inclure les paramètres)

json = url_get(url, limit=5, author='Confucius') # inclusion des deux contraintes dans la requête GET
for quote in json['results']: # les citations sont regroupées sous la clé 'results' du dictionnaire
  print(f"● {quote['content']} ({quote['author']})") # extraction des champs 'content' et 'author'

● The superior man understands what is right; the inferior man understands what will sell. (Confucius)
● Wisdom, compassion, and courage are the three universally recognized moral qualities of men. (Confucius)
● To be wronged is nothing unless you continue to remember it. (Confucius)
● If you look into your own heart, and you find nothing wrong there, what is there to worry about? What is there to fear? (Confucius)
● Study the past, if you would divine the future. (Confucius)


Le même mécanisme de requêtes GET permet, par exemple, d'utiliser des traducteurs automatiques. On utilise ici le site **`mymemory.translate.net`** qui, certes ne fournit pas les traductions les plus précises, mais possède l'avantage de pouvoir être utilisé sans abonnement, et même sans créer de compte utilisateur spécifique :

In [20]:
def translate(text, pair='en|fr'):
  """translate provided text in other language according to 'pair' (english to french, by default)"""
  url = 'https://api.mymemory.translated.net/get' # set URL for translation website
  json = url_get(url, q=text, langpair=pair) # set parameters of GET request used for translation
  return json['responseData']['translatedText'] # extract and return translated text

In [21]:
for quote in json['results']:
  print(f"● {translate(quote['content'])} ({quote['author']})") # traduction de chaque citation

● L'homme supérieur comprend ce qui est juste ; l'homme inférieur comprend ce qui se vendra. (Confucius)
● La sagesse, la compassion et le courage sont les trois qualités morales universellement reconnues des hommes. (Confucius)
● Être lésé n'est rien à moins que vous ne continuiez à vous en souvenir. (Confucius)
● Si vous regardez dans votre propre cœur, et que vous ne trouvez rien de mal là-dedans, de quoi s'inquiéter ? Qu'y a-t-il à craindre ? (Confucius)
● Étudiez le passé, si vous voulez deviner l'avenir. (Confucius)


---
### 2 - Utilisation des requêtes POST

Comme on l'a fait plus haut pour les requêtes GET, on va créer une fonction **`url_post`** qui va permettre de regrouper la séquence des opérations habituellement utilisées dans une requête POST :

In [22]:
def url_post(url, **args):
  """apply POST request on provided URL, using optional arguments stored as arbitrary keyword arguments"""
  headers = {'User-Agent':'Mozilla/5.0'} # use Mozilla/Firefox user agent (anti-scraping filter)
  data = requests.post(url, headers=headers, data=args) # connect to provided URL and try to get data
  data.raise_for_status() # raise HTTPError exception when connection or server error
  return data.json() # return data in JSON format when status is OK

Une des utilisations les plus classiques des requêtes POST, est de pouvoir fournir à un site d'accès restreint, les paramètres de son compte utilisateur : **nom de login** et **mot de passe**. On peut tester ce mécanisme sur le site **reqres.in** qui est un service en ligne permettant d'expérimenter diverses requêtes HTTP :

In [23]:
url = 'https://reqres.in/api/login' # test de connexion avec identification

credentials = dict(email='eve.holt@reqres.in', password='p@$$w0RD!') # paramètres de connexion
json = url_post(url, **credentials) # inclusion du dictionnaire des paramètres dans la requête POST
print(json) # le site renvoie un JSON contenant un token unique pour chaque compte utilisateur valide

{'token': 'QpwL5tke4Pnpja7X4'}


Les requêtes POST permettent de transmettre des données arbitrairement complexes à un site web, sous la forme d'une chaîne de caractères au format JSON. Cette chaîne peut évidemment être cryptée si on doit transmettre des informations sensibles. Ce principe est utilisé, par exemple, sur tous les sites de vente en ligne, et plus généralement sur tous les sites nécessitant une authentification par l'utilisateur.

A titre d'exemple, on va utiliser le site **Open-Meteo.com** pour obtenir les prévisions météo à 7 jours sur Bordeaux :

In [24]:
url = 'https://api.open-meteo.com/v1/forecast' # prévisions météo

queries = 'temperature_2m_min temperature_2m_max windspeed_10m_max'.split() # données à récupérer
json = url_post(url, latitude=44.84, longitude=-0.56, daily=queries) # inclusion des trois paramètres
print(json, '\n') # les données (complexes) sont formatées sous la forme d'une chaîne JSON multi-lignes
data = [json['daily'][query] for query in ['time']+queries] # extraction des données souhaitées
for date, tmin, tmax, wind in zip(*data): # boucle d'affichage
  print(f"● {date} : min = {tmin:4} °C / max = {tmax:4} °C / wind = {wind:4} km/h")

{'latitude': 44.84, 'longitude': -0.5600002, 'generationtime_ms': 0.09000301361083984, 'utc_offset_seconds': 0, 'timezone': 'GMT', 'timezone_abbreviation': 'GMT', 'elevation': 11.0, 'daily_units': {'time': 'iso8601', 'temperature_2m_min': '°C', 'temperature_2m_max': '°C', 'windspeed_10m_max': 'km/h'}, 'daily': {'time': ['2024-12-03', '2024-12-04', '2024-12-05', '2024-12-06', '2024-12-07', '2024-12-08', '2024-12-09'], 'temperature_2m_min': [7.7, 4.1, 4.2, 13.9, 6.8, 4.2, 3.6], 'temperature_2m_max': [11.2, 10.2, 14.9, 16.7, 15.4, 9.5, 6.5], 'windspeed_10m_max': [3.7, 6.5, 17.1, 22.2, 23.5, 21.7, 24.7]}} 

● 2024-12-03 : min =  7.7 °C / max = 11.2 °C / wind =  3.7 km/h
● 2024-12-04 : min =  4.1 °C / max = 10.2 °C / wind =  6.5 km/h
● 2024-12-05 : min =  4.2 °C / max = 14.9 °C / wind = 17.1 km/h
● 2024-12-06 : min = 13.9 °C / max = 16.7 °C / wind = 22.2 km/h
● 2024-12-07 : min =  6.8 °C / max = 15.4 °C / wind = 23.5 km/h
● 2024-12-08 : min =  4.2 °C / max =  9.5 °C / wind = 21.7 km/h
● 202

<h2 style="padding:16px; color:#FFF; background:#07D">C - Beautiful Soup</h2>

Comme on l'a noté dans la section A, lorsque les données à récupérer dans une page HTML sont regroupées dans une balise **`<table>...</table>`**, la fonction **`read_html`** de **pandas** est parfaitement adaptée. Parfois, il faut effectuer quelques manipulations en post-traitement pour supprimer ou réorganiser certaines colonnes, mais cela ne nécessite habituellement pas plus de 2 ou 3 lignes de code. Par contre, lorsque les données ne sont pas structurées sous forme d'une tables HTML, mais en utilisant des combinaisons de balises **`<div>...</div>`** et **`<span>...</span>`**, il faudra utiliser des packages spécialement développés pour effectuer un ***web scraping*** finement paramétrable. De même, lorsque les données sont réparties sur de nombreuses pages au sein d'un même site, il faudra mettre en oeuvre des outils automatiques pour réaliser le ***web crawling*** afin de ne pas devoir récupérer les données manuellement, page par page.

Il existe trois principales bibliothèques en Python pour effectuer ce type de tâches : [**BeautifulSoup**](https://beautiful-soup-4.readthedocs.io/en/latest), [**Scrapy**](https://docs.scrapy.org/en/latest) et [**Selenium**](https://selenium-python.readthedocs.io). Dans cette section, nous allons simplement explorer quelques fonctionnalités de **BeautifulSoup** qui est le plus simple à mettre en oeuvre parmi ces trois packages. Pour certaines extractions de données avancées, vous serez peut-être amenés à utiliser les deux autres, c'est pour cela que j'ai mis les liens vers la documentation correspondante sur le site **ReadTheDocs**.

Le package **`BeautifulSoup`** ne fait pas partie de la bibliothèque standard de Python, elle nécessite donc une installation spécifique à l'aide de l'outil **`pip`**, mais comme le package **`requests`**, sa très grande popularité dans le domaine des Sciences des Données, fait qu'elle est directement incluse dans la distribution **Anaconda**. Il faut noter qu'à l'inverse, les packages **Scrapy** et **Selenium** ne sont pas inclus par défaut dans **Anaconda**, du fait de leur utilisation plus spécialisée.

In [25]:
from bs4 import BeautifulSoup as bs # import de la fonction 'BeautifulSoup' du package bs4'

Pour tester les fonctionnalités de la bibliothèque, on va utiliser le site **quotes.toscrape.com** qui regroupe de nombreuses citations de personnes célèbres, et qui a été spécifiquement créé comme site d'entraînement pour le web scraping.

La première étape consiste à récupérer le code source HTML de la page, avec la fonction **`load`** :

In [26]:
url = 'https://quotes.toscrape.com' # dictionnaire de citations

html = load(url, split='') # lecture du code source de la page HTML
print(html[723:1142]) # affichage d'un extrait de la page pour comprendre la structure utilisée

    <div class="quote" itemscope itemtype="http://schema.org/CreativeWork">
        <span class="text" itemprop="text">“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”</span>
        <span>by <small class="author" itemprop="author">Albert Einstein</small>
        <a href="/author/Albert-Einstein">(about)</a>
        </span>
        <div class="tags">


Ensuite, on transmet ce code à la fonction **`bs`** en lui précisant de l'analyser avec un **parser** HTML (**BeautifulSoup** peut également s'utiliser sur du code XML). La fonction renvoie une structure appelée **`soup`** qui contient, en plus du code brut, des informations hiérarchiques sur les différentes balises utilisées dans ce code.

In [27]:
soup = bs(html, 'html.parser') # on transforme le code HTML en "soupe" avec la fonction 'bs'
print(str(soup)[490:868]) # les lignes ont été 'stripées', donc il faut ajuster la position de l'extrait

<div class="quote" itemscope="" itemtype="http://schema.org/CreativeWork">
<span class="text" itemprop="text">“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”</span>
<span>by <small class="author" itemprop="author">Albert Einstein</small>
<a href="/author/Albert-Einstein">(about)</a>
</span>
<div class="tags">


En analysant le code HTML (on n'a mis qu'un petit extrait ci-dessus pour économiser de la place), on constate que chaque citation est structurée de manière similaire :
- Toutes les données d'une même citation sont placées derrière une balise **`<div class="quote"...>`**
- Le texte de la citation est placée derrière une balise **`<span class="text"...>`**
- L'auteur de la citation est placée derrière une balise **`<small class="author"...>`**

On peut donc fournir ces indications à **BeautifulSoup** pour qu'il puisse effectuer l'extraction des données souhaitées :

In [28]:
for quote in soup.find_all('div', class_='quote'): # parcours de toutes les balises <div class='quote'>
  text = quote.find('span', class_='text').text # extraction de la balise <span class='text'>...</span>
  author = quote.find('small', class_='author').text # idem pour <small class='author'>...</small>
  print(f"● {text} ({author})") # affichage de la citation récupérée en anglais
  print(f"● {translate(text)} ({author})\n") # et traduction en français avec la fonction 'translate'

● “The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.” (Albert Einstein)
● « Le monde tel que nous l'avons créé est un processus de notre pensée. Il ne peut pas être changé sans changer notre façon de penser. » (Albert Einstein)

● “It is our choices, Harry, that show what we truly are, far more than our abilities.” (J.K. Rowling)
● « Ce sont nos choix qui montrent qui nous sommes vraiment, bien plus que nos capacités. » (J.K. Rowling)

● “There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.” (Albert Einstein)
● “Il n’y a que deux façons de vivre sa vie ; penser que rien n’est un miracle ou penser que tout est un miracle.” (Albert Einstein)

● “The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.” (Jane Austen)
● "La personne, que ce soit un gentleman ou une dame, qui n'a pas de plaisir dans un bon 

En analysant le code HTML plus finement, on peut identifier à la fin de chaque page, un lien permettant de se rendre à la page suivante. On peut donc utiliser ce lien pour faire un **web crawling** sur l'ensemble du site, et récupérer ainsi toutes les citations, ou comme on va le faire ci-dessous, établir un classement du nombre de citations par auteur :

In [29]:
base = url = 'https://quotes.toscrape.com/' # stockage de l'URL de base, afin de construire chaque URL
histo = {} # initialisation d'un histogramme pour compter les citations par auteur
while url: # boucle sur les différentes pages du site
  html = load(url, split='') # lecture du code HTML de la page courante
  soup = bs(html, 'html.parser') # conversion du code HTML en "soupe"
  for quote in soup.find_all('div', class_='quote'): # parcours des balises <div class='quote'>
    author = quote.find('small', class_='author').text # idem pour <small class='author'>...</small>
    histo[author] = histo.get(author,0) + 1 # mise à jour de l'histogramme
    newpage = soup.find('li', class_='next') # recherche du lien vers la page suivante
    url = base + newpage.find('a')['href'] if newpage else '' # mise à jour de l'URL pour la page suivante
for name,n in sorted(histo.items(), key=lambda x:(-x[1],x[0])): print(name, ':', n, end=' / ')

Albert Einstein : 10 / J.K. Rowling : 9 / Marilyn Monroe : 7 / Dr. Seuss : 6 / Mark Twain : 6 / C.S. Lewis : 5 / Jane Austen : 5 / Bob Marley : 3 / Charles Bukowski : 2 / Eleanor Roosevelt : 2 / Ernest Hemingway : 2 / George R.R. Martin : 2 / Mother Teresa : 2 / Ralph Waldo Emerson : 2 / Suzanne Collins : 2 / Alexandre Dumas fils : 1 / Alfred Tennyson : 1 / Allen Saunders : 1 / André Gide : 1 / Ayn Rand : 1 / Charles M. Schulz : 1 / Douglas Adams : 1 / E.E. Cummings : 1 / Elie Wiesel : 1 / Friedrich Nietzsche : 1 / Garrison Keillor : 1 / George Bernard Shaw : 1 / George Carlin : 1 / George Eliot : 1 / Harper Lee : 1 / Haruki Murakami : 1 / Helen Keller : 1 / J.D. Salinger : 1 / J.M. Barrie : 1 / J.R.R. Tolkien : 1 / James Baldwin : 1 / Jim Henson : 1 / Jimi Hendrix : 1 / John Lennon : 1 / Jorge Luis Borges : 1 / Khaled Hosseini : 1 / Madeleine L'Engle : 1 / Martin Luther King Jr. : 1 / Pablo Neruda : 1 / Stephenie Meyer : 1 / Steve Martin : 1 / Terry Pratchett : 1 / Thomas A. Edison : 

<h2 style="padding:0 0 8px; margin:0 -20px; color:#FFF; background:#07D; text-align:right">● ● ●</h3>